# StressSense — data exploration

A short walkthrough of the raw Empatica E4 signals and how they become labeled
windows. Run with the project environment from the `notebooks/` folder. Outputs
are cleared in the repo; run all cells to regenerate the figures.

Sections: dataset layout → raw signals for one session → how labels are derived
from tags → the final window cache.

In [ ]:
import os, sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from dataset import (load_config, session_channels, read_tags,
                     session_regions, load_cache)

cfg = load_config('../config.yaml')
root = os.path.join(cfg['data']['dataset_root'], 'Wearable_Dataset')
channels = cfg['signals']['channels']
target_fs = cfg['signals']['target_fs']
print('dataset root:', root)
print('channels:', channels, '| target_fs:', target_fs)

## Dataset layout
Three activity folders, each with one subfolder per session.

In [ ]:
for activity in ['STRESS', 'AEROBIC', 'ANAEROBIC']:
    subs = sorted(os.listdir(os.path.join(root, activity)))
    print(f'{activity:10s} {len(subs):2d} sessions: {subs}')

## Raw signals for one stress session
Each channel is resampled onto a common 4 Hz grid. Red lines are protocol tags
(button presses); shaded spans are the documented mental-stress tasks.

In [ ]:
activity, subject = 'STRESS', 'S01'
sess = os.path.join(root, activity, subject)
X, valid, t0 = session_channels(sess, channels, target_fs)
tags = read_tags(os.path.join(sess, 'tags.csv'), t0)
duration = X.shape[1] / target_fs
regions = session_regions(activity, subject, tags, duration, cfg)
t = np.arange(X.shape[1]) / target_fs

fig, axes = plt.subplots(len(channels), 1, figsize=(12, 8), sharex=True)
for ax, ch, row in zip(axes, channels, X):
    ax.plot(t, row, lw=0.8)
    ax.set_ylabel(ch)
    for tag in tags:
        ax.axvline(tag, color='r', lw=0.5, alpha=0.5)
    for label, a, b in regions:
        if label == 'stress':
            ax.axvspan(a, b, color='orange', alpha=0.2)
axes[-1].set_xlabel('time (s)')
axes[0].set_title(f'{activity}/{subject} — orange = stress tasks')
plt.tight_layout(); plt.show()

## Derived labels for this session
`session_regions` turns the tags into labeled (label, start_s, end_s) spans.

In [ ]:
for label, a, b in regions:
    print(f'{label:10s} {a:7.1f}s -> {b:7.1f}s  ({b - a:5.1f}s)')

## The window cache
After running `python dataset.py`, the whole dataset is a single array of
windows. Class balance and a quick look at how the channels differ by class.

In [ ]:
cache = load_cache(cfg['data']['cache_path'])
X, y = cache['X'], cache['y']
classes = [str(c) for c in cache['classes']]
print('windows:', X.shape, '| subjects:', len(set(cache['subject'].tolist())))
counts = [int((y == i).sum()) for i in range(len(classes))]
plt.bar(classes, counts); plt.ylabel('windows'); plt.title('Class balance'); plt.show()

In [ ]:
# Mean of each channel within a window, grouped by class. The ACC (motion)
# channel is what cleanly separates the seated classes from the exercise ones.
fig, axes = plt.subplots(1, len(channels), figsize=(14, 3.5))
for ax, ci in zip(axes, range(len(channels))):
    per_window_mean = X[:, ci, :].mean(axis=1)
    data = [per_window_mean[y == k] for k in range(len(classes))]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(classes) + 1)); ax.set_xticklabels(classes, rotation=30)
    ax.set_title(channels[ci])
plt.tight_layout(); plt.show()

**Takeaways.** Motion intensity (ACC) is near zero for `baseline`/`stress` and
high for `aerobic`/`anaerobic`, which is why the model never confuses stress with
exercise. The harder separations — `baseline` vs `stress`, and `aerobic` vs
`anaerobic` — rely on subtler differences in EDA, HR and motion variability.